# Train the Qwen3.5-9B WhatsApp LoRA on Colab

Trains the bf16 LoRA adapter from `whatsapp_train.jsonl` and uploads it to a **private** HF model repo, which the Space `vasu1712/qwen3.5-LoRA` then loads via its `ADAPTER_ID` variable.

**Runtime**: `Runtime > Change runtime type > A100 GPU` (recommended, ~1–1.5 h ≈ 30–45 compute units). L4 24 GB works with the fallback command in the training cell. **T4 / free tier cannot run this** (no bf16, not enough VRAM).

Run cells top to bottom. Keep the tab open during training — Colab disconnects idle sessions.

In [ ]:
# 1. GPU guard — fail fast on an unsuitable GPU before installing anything
import torch
assert torch.cuda.is_available(), "No GPU. Runtime > Change runtime type > A100 (or L4)."
name = torch.cuda.get_device_name(0)
cap = torch.cuda.get_device_capability(0)
vram = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f"{name} | compute capability {cap} | {vram:.0f} GB VRAM")
assert cap >= (8, 0), (
    f"{name} has no bf16 support (needs Ampere or newer). "
    "Switch the runtime to A100 or L4 — T4/P100 cannot train Qwen3.5."
)
if vram < 30:
    print("NOTE: 24 GB-class GPU (L4) — use the L4 fallback command in the training cells.")

In [ ]:
# 2. Install deps (mirrors requirements-train.txt; transformers from git main is
#    REQUIRED for Qwen3.5 — no bitsandbytes: QLoRA/4-bit is not recommended for it)
%pip install -q "transformers[serving] @ git+https://github.com/huggingface/transformers.git@main" "trl>=0.12" "peft>=0.11" "accelerate>=0.30" "datasets>=2.19"

In [ ]:
# 3. Upload the two files from your project folder:
#    - train_qlora.py        (the bf16 LoRA training script)
#    - whatsapp_train.jsonl  (private training data — uploaded directly, never via git)
from google.colab import files
uploaded = files.upload()
import os
assert os.path.exists("train_qlora.py"), "train_qlora.py missing — upload it"
assert os.path.exists("whatsapp_train.jsonl"), "whatsapp_train.jsonl missing — upload it"
print("files in place")

In [ ]:
# 4. Log in to Hugging Face — paste a WRITE token (huggingface.co/settings/tokens)
from huggingface_hub import notebook_login
notebook_login()

## 5. Smoke test (~10 min) — the go/no-go gate before the real run

Confirm in the output: `trainable params: ...` is **non-zero and roughly 1–2%** of all params (proves LoRA attached to the hybrid layers), and the loss **decreases** over the 20 steps. Only then run the full training cell.

In [ ]:
!MAX_STEPS=20 TRAIN_FILE=whatsapp_train.jsonl OUTPUT_DIR=adapter python train_qlora.py

In [ ]:
# 6. Full run (~645 steps; ~1–1.5 h on A100). Keep the tab open.
!TRAIN_FILE=whatsapp_train.jsonl OUTPUT_DIR=adapter python train_qlora.py

# L4 24 GB fallback — use INSTEAD of the line above (slower, shorter context):
# !BATCH_SIZE=1 GRAD_ACCUM=16 MAX_LEN=1024 TRAIN_FILE=whatsapp_train.jsonl OUTPUT_DIR=adapter python train_qlora.py

In [ ]:
# 7. Upload the adapter to a PRIVATE model repo (trained on real chats)
from huggingface_hub import HfApi
repo = "vasu1712/qwen3.5-whatsapp-lora"
api = HfApi()
api.create_repo(repo, repo_type="model", private=True, exist_ok=True)
api.upload_folder(folder_path="adapter", repo_id=repo, repo_type="model")
print(f"Done: https://huggingface.co/{repo} (private)")

## 8. Test it on the Space

1. Push the updated Space code if not done yet (`git push space main` from the project).
2. Space **Settings → Variables and secrets**: variable `ADAPTER_ID = vasu1712/qwen3.5-whatsapp-lora`, secret `HF_TOKEN` = a **read** token (the adapter repo is private). Restart the Space.
3. Header should read `Base Qwen/Qwen3.5-9B · Adapter vasu1712/qwen3.5-whatsapp-lora`.
4. **Human-touch protocol**: append to the system prompt box — *"Be personable and human — acknowledge feelings, remember context, use casual natural language."* Probe with emotional prompts ("we just had a baby and I'm stressed about money"; mid-chat: "sorry, my mom was in hospital") and a 4–5-turn conversation (must not re-greet). Temperature 0.7–0.9.
5. **A/B**: clear `ADAPTER_ID`, restart, repeat the identical prompts on the bare base — the transcript difference is what your adapter learned.